# Chicago Crime: Exploratory Analysis (2019–2025)

**Questions this notebook answers**

1. How has crime changed since 2019, including the 2020 pandemic dip and rebound?
2. When does crime happen (hour and weekday), and does that differ by crime type?
3. Which crimes are most common, and how likely is each to end in an arrest?
4. Has the arrest rate changed over time?
5. Where is crime concentrated across Chicago's 77 community areas?
6. What role do domestic incidents and location types play?

**Data:** City of Chicago, *Crimes – 2001 to Present* (public). Download with `python scripts/download_data.py`.

## 1. Setup and load

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.cleaning import clean, quality_report

DATA = Path(os.getenv("CRIME_DATA", ROOT / "data" / "crimes_2019_2025.parquet"))
IMAGES = ROOT / "images"
IMAGES.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
ACCENT = "#2b6cb0"

def save(fig, name):
    """Save every chart so the README can show them."""
    fig.savefig(IMAGES / f"{name}.png", bbox_inches="tight")

raw = pd.read_parquet(DATA)
print(f"{len(raw):,} rows, {raw.shape[1]} columns")
raw.head()

## 2. Data quality

Look at the problems before fixing them, so each cleaning step has a reason.

In [ ]:
quality_report(raw)

In [ ]:
df, log = clean(raw)
pd.Series(log, name="count").to_frame()

**What the cleaning does** (details in `src/cleaning.py`)

- Drops duplicate record IDs and rows with no date or crime type.
- Converts the API's text values to real dates, booleans and numbers.
- Sets coordinates outside Chicago to missing. The incident still counts; it just can't be mapped.
- Treats community area `0` as missing, since it isn't a real area.
- Adds features: month, hour, weekday, weekend flag, season.

In [ ]:
# Many incidents are logged at exactly 00:00 when the real time is unknown.
# That would create a fake midnight spike in the hour-of-day analysis, so check it first.
midnight_share = df["time_is_midnight"].mean()
print(f"Logged at exactly 00:00: {midnight_share:.1%} of incidents")
df.groupby("primary_type")["time_is_midnight"].mean().sort_values(ascending=False).head(8).map("{:.1%}".format)

## 3. Trend since 2019

In [ ]:
monthly = df.groupby("month").size()

fig, ax = plt.subplots(figsize=(11, 4))
monthly.plot(ax=ax, color=ACCENT, linewidth=2)
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2020-06-01"), color="grey", alpha=0.15)
ax.annotate("Stay-at-home order", xy=(pd.Timestamp("2020-03-15"), monthly.max() * 0.97), fontsize=9)
ax.set(title="Reported incidents per month", xlabel="", ylabel="Incidents")
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
save(fig, "01_monthly_trend")
plt.show()

In [ ]:
yearly = df.groupby("year").size().rename("incidents").to_frame()
yearly["change_vs_prev_pct"] = (yearly["incidents"].pct_change() * 100).round(1)
yearly

In [ ]:
# Different crimes moved very differently through the pandemic. Index each to 2019 = 100.
top6 = df["primary_type"].value_counts().head(6).index
by_type = df[df["primary_type"].isin(top6)].groupby(["year", "primary_type"]).size().unstack()
indexed = (100 * by_type / by_type.iloc[0]).round(0)

fig, ax = plt.subplots(figsize=(10, 4.5))
indexed.plot(ax=ax, marker="o", linewidth=2)
ax.axhline(100, color="black", linewidth=0.8, linestyle="--")
ax.set(title="Top crime types, indexed to 2019 = 100", xlabel="", ylabel="Index")
ax.legend(title="", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
save(fig, "02_types_indexed")
plt.show()
indexed

## 4. When does crime happen?

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
timed = df[~df["time_is_midnight"]]          # exclude unknown times (see section 2)
heat = timed.pivot_table(index="weekday", columns="hour", values="id", aggfunc="count").reindex(order)

fig, ax = plt.subplots(figsize=(12, 3.8))
sns.heatmap(heat, cmap="Blues", ax=ax, cbar_kws={"label": "Incidents"})
ax.set(title="Incidents by weekday and hour (unknown times excluded)", xlabel="Hour of day", ylabel="")
save(fig, "03_weekday_hour_heatmap")
plt.show()

In [ ]:
# Peak hour differs a lot by crime type
peak = (timed.groupby(["primary_type", "hour"]).size()
        .groupby(level=0).idxmax().map(lambda x: x[1]).rename("peak_hour"))
volume = timed["primary_type"].value_counts().rename("incidents")
pd.concat([volume, peak], axis=1).head(10)

## 5. What crimes, and how often do they end in an arrest?

In [ ]:
summary = (df.groupby("primary_type")
             .agg(incidents=("id", "count"), arrest_rate=("arrest", "mean"), domestic_share=("domestic", "mean"))
             .sort_values("incidents", ascending=False))
summary["share_of_all_pct"] = (100 * summary["incidents"] / summary["incidents"].sum()).round(1)
top10 = summary.head(10)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
top10["incidents"].plot.barh(ax=axes[0], color=ACCENT)
axes[0].set(title="Incidents", xlabel="", ylabel="")
axes[0].xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
(top10["arrest_rate"] * 100).plot.barh(ax=axes[1], color="#dd6b20")
axes[1].set(title="Arrest rate (%)", xlabel="")
axes[0].invert_yaxis()
fig.suptitle("Top 10 crime types", y=1.02)
save(fig, "04_top_types_arrest_rate")
plt.show()
top10.style.format({"incidents": "{:,}", "arrest_rate": "{:.1%}", "domestic_share": "{:.1%}"})

**How to read the arrest rates:** crimes like narcotics or weapons violations are often discovered *by* police during a stop, so they almost always come with an arrest. Theft and burglary are reported later by victims, with no suspect present. A low arrest rate is not automatically a policing failure; it reflects how each crime comes to light.

In [ ]:
arrest_trend = df.groupby("year")["arrest"].mean().mul(100).round(1).rename("arrest_rate_pct")
fig, ax = plt.subplots(figsize=(8, 3.5))
arrest_trend.plot(ax=ax, marker="o", color="#dd6b20", linewidth=2)
ax.set(title="Share of incidents ending in an arrest", xlabel="", ylabel="%", ylim=(0, None))
save(fig, "05_arrest_rate_trend")
plt.show()
arrest_trend.to_frame()

## 6. Where is crime concentrated?

In [ ]:
areas = (df.dropna(subset=["community_name"])
           .groupby("community_name")
           .agg(incidents=("id", "count"), arrest_rate=("arrest", "mean"))
           .sort_values("incidents", ascending=False))
areas["share_pct"] = 100 * areas["incidents"] / areas["incidents"].sum()
areas["cumulative_pct"] = areas["share_pct"].cumsum()

top_n = (areas["cumulative_pct"] <= 50).sum() + 1
print(f"The top {top_n} of 77 community areas account for half of all incidents.")

fig, ax = plt.subplots(figsize=(9, 5.5))
areas.head(15)["incidents"].plot.barh(ax=ax, color=ACCENT)
ax.invert_yaxis()
ax.set(title="Top 15 community areas by incidents", xlabel="Incidents", ylabel="")
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
save(fig, "06_top_community_areas")
plt.show()

**Limitation:** these are raw counts, not rates per resident. Large or busy areas (the Loop, Near North Side) draw many visitors, so high counts there don't mean residents face higher risk. A per-capita comparison would need census population data, a natural next step.

In [ ]:
geo = df.dropna(subset=["latitude", "longitude"])
sample = geo.sample(min(60_000, len(geo)), random_state=42)

fig, ax = plt.subplots(figsize=(6.5, 8))
ax.scatter(sample["longitude"], sample["latitude"], s=0.3, alpha=0.25, color=ACCENT)
ax.set(title=f"Incident locations (random sample of {len(sample):,})", xlabel="", ylabel="")
ax.set_aspect(1.3)
ax.set_xticks([]); ax.set_yticks([])
save(fig, "07_incident_map")
plt.show()

## 7. Domestic incidents and location types

In [ ]:
dom = summary[summary["incidents"] >= 1000].sort_values("domestic_share", ascending=False).head(8)
dom[["incidents", "domestic_share"]].style.format({"incidents": "{:,}", "domestic_share": "{:.1%}"})

In [ ]:
loc = df["location_description"].value_counts()
loc_share = (100 * loc / loc.sum()).round(1).head(10).rename("share_pct").to_frame()
loc_share["arrest_rate_pct"] = (df.groupby("location_description")["arrest"].mean()
                                 .reindex(loc_share.index) * 100).round(1)
loc_share

## 8. Seasonality

In [ ]:
season_order = ["Winter", "Spring", "Summer", "Fall"]
seasonal = (df[df["primary_type"].isin(top6)]
              .groupby(["season", "primary_type"]).size().unstack()
              .reindex(season_order))
seasonal_pct = (100 * seasonal / seasonal.sum()).round(1)   # share of each type's incidents by season

fig, ax = plt.subplots(figsize=(9, 4))
seasonal_pct.T.plot.bar(ax=ax, width=0.8)
ax.axhline(25, color="black", linewidth=0.8, linestyle="--")
ax.set(title="Share of each crime type by season (25% = no seasonality)", xlabel="", ylabel="%")
ax.legend(title="", frameon=False)
plt.xticks(rotation=30, ha="right")
save(fig, "08_seasonality")
plt.show()

## 9. Summary numbers

Computed automatically so the findings below always match the data.

In [ ]:
y = df.groupby("year").size()
first, last = y.index.min(), y.index.max()
facts = {
    "Incidents analyzed": f"{len(df):,} ({first}-{last})",
    f"Change {first} to {last}": f"{(y[last] / y[first] - 1):+.1%}",
    "Overall arrest rate": f"{df['arrest'].mean():.1%}",
    f"Arrest rate {first} vs {last}": f"{arrest_trend[first]}% vs {arrest_trend[last]}%",
    "Most common crime": f"{summary.index[0]} ({summary['share_of_all_pct'].iloc[0]}% of all)",
    "Domestic share of all incidents": f"{df['domestic'].mean():.1%}",
    "Community areas holding half of incidents": f"{top_n} of 77",
    "Busiest hour (known times)": f"{timed['hour'].value_counts().idxmax()}:00",
    "Logged at midnight (unknown time)": f"{midnight_share:.1%}",
}
pd.Series(facts, name="value").to_frame()

## 10. Key findings

> Write these after running the notebook, using the numbers in section 9 and the charts. Each finding = a number + what it means for someone making decisions (police deployment, city budget, residents). Example format:
> - **Theft is X% of all incidents but ends in arrest only Y% of the time.** Prevention (lighting, cameras, awareness) likely pays off more than enforcement for this crime.

1.
2.
3.
4.
5.

## Next steps

- Per-capita rates using census population by community area.
- Join with weather data to test the "hot days, more crime" idea.
- A forecasting model for monthly incidents by area.